# 11 - V2.3 Dual-Track Quick: Tweedie direct public-LB + bridge robuste

Objectif:
- comparer proprement le **track robuste V2/V2.2** avec un **track direct Tweedie CatBoost** (random KFold + scale/blend),
- diagnostiquer `overfit vs OOD vs queue`,
- produire 2 soumissions V2.3 (`robust`, `lb_challenger`) sans toucher aux fichiers existants.

Tous les outputs sont écrits dans `artifacts/v2_3_dualtrack_quick/`.


## Clarification importante (OOF vs Kaggle public)

- `~542` = RMSE local OOF (train/validation)
- `~1365.5` = Kaggle public (hidden test, 1/3 de l'éval)
- ces métriques **ne sont pas directement comparables**

Ce notebook sépare explicitement:
- `overfitting CV`,
- `OOD / shift`,
- `sous-modélisation de la queue`,
- `heuristiques public-LB` (scale/blend).


In [2]:
import sys
import json
from pathlib import Path
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR
for _candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (_candidate / "src").exists() and (_candidate / "data").exists():
        ROOT = _candidate
        break
else:
    raise RuntimeError("Project root not found. Expected folders: src/ and data/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACTS_DIR = ROOT / "artifacts"

from src.insurance_pricing.experiments.quick import dualtrack as wf
import src.insurance_pricing.training as v2

# =========================
# Config (quick / 1-2h)
# =========================
SEED = 42
KAGGLE_PUBLIC_RMSE_USER = 1365.5
EXTERNAL_STEP7_SUBMISSION_PATH = None  # ex: r"C:\...\submission_step7_tweedie_p12_x1.3.csv"

RUN_DIRECT_TWEEDIE_REPRO = False
RUN_DIRECT_TWEEDIE_V2_SPLITS = False
RUN_BLEND_SWEEP = False
RUN_SHAKEUP_QUICK = False
QUICK_MODE = True

N_SPLITS_RANDOM = 5
TWEEDIE_POWERS = [1.2, 1.4]
SCALE_SWEEP = [1.0, 1.1, 1.2, 1.3]
BLEND_WEIGHTS = [0.2, 0.4, 0.6, 0.8]
TOP_POWERS_FOR_V2_EVAL = 1 if QUICK_MODE else 2

CATBOOST_ITER = 12000
CATBOOST_LR = 0.03
CATBOOST_DEPTH = 8
OD_WAIT = 300
N_SIM_SHAKEUP = 300 if QUICK_MODE else 800

ARTIFACT_V23 = wf.ensure_v23_dir(ROOT)
print("ROOT:", ROOT)
print("ARTIFACT_V23:", ARTIFACT_V23)


ROOT: c:\Users\icemo\Downloads\Calcul-prime-d-assurance
ARTIFACT_V23: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_3_dualtrack_quick


In [2]:
# 1) Chargement des artefacts existants (lecture seule)
ctx = wf.load_existing_artifacts(ROOT)
print("v1_run_registry:", pd.DataFrame(ctx.get("v1_run_registry")).shape)
print("v2_run_registry:", pd.DataFrame(ctx.get("v2_run_registry")).shape)
print("v2_2_quick_retrain_registry:", pd.DataFrame(ctx.get("v22_quick_retrain_registry")).shape)
print("ds_metrics:", pd.DataFrame(ctx.get("ds_metrics")).shape)
print("v2_2 gap summary:", ctx.get("v22_quick_gap_summary", {}))


v1_run_registry: (78, 14)
v2_run_registry: (81, 18)
v2_2_quick_retrain_registry: (12, 62)
ds_metrics: (1, 20)
v2_2 gap summary: {'kaggle_public_rmse_user': 1366.0, 'note_metric_non_comparable': True, 'n_runs_analyzed': 10, 'global_unseen_focus_flag': 0, 'dominant_hypothesis_top_run': 'Gap domine par sous-modelisation de la queue', 'dominant_hypothesis_counts': {'Gap domine par sous-modelisation de la queue': 10}}


## 2) Bridge V1 / V2 / V2.2 + comparatif de distributions de soumissions

Décision pratique:
- on fixe un pont de comparaison **local + distributionnel**,
- on évite de piloter le cycle uniquement au score public.


In [3]:
bridge_summary_df = wf.build_bridge_summary(ctx, kaggle_public_rmse=KAGGLE_PUBLIC_RMSE_USER)
bridge_summary_df.to_csv(ARTIFACT_V23 / "run_bridge_summary.csv", index=False)
print("saved:", ARTIFACT_V23 / "run_bridge_summary.csv")

pred_distribution_existing_df = wf.build_pred_distribution_compare(ctx)
display(bridge_summary_df)
display(pred_distribution_existing_df)


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_3_dualtrack_quick\run_bridge_summary.csv


,model_version,track,run_id,rmse_primary_time,auc_freq,brier_freq,rmse_sev_pos,q99_ratio_pos,rmse_secondary_group,rmse_aux_blocked5,rmse_split_std,q95_ratio_pos,rmse_prime_top1pct,selection_score_quick,kaggle_public_rmse_user
0,v1_best_local,v1_two_part,catboost|cb_c1|42|classic|none,542.672558,0.650693,0.054100,1476.722719,0.309563,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,v2_selected_top,v2_two_part,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,543.897946,0.608750,0.054531,1462.342755,0.328733,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ds_diagnostic_run,v2_two_part,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,545.929482,0.594651,0.054613,1662.270082,0.328630,NaN,NaN,NaN,0.476975,4548.227515,NaN,1365.5
3,v2_2_quick_robust,v2_2_quick,robust_v2|catboost|two_part_tweedie|1.5|cb_v2_...,553.388328,NaN,NaN,NaN,0.998603,550.834467,557.692024,2.829804,0.916466,4532.662229,594.507678,NaN
4,v2_2_quick_challenger,v2_2_quick,base_v2|xgboost|two_part_tweedie|1.5|xgb_v2_c1...,553.393912,NaN,NaN,NaN,0.988316,560.638507,556.975609,2.957655,0.924568,4545.784708,598.652424,NaN


,name,exists,n,mean,std,q90,q95,q99,max,share_zero
0,submission_v1,1,50000,97.606838,62.347925,172.878272,201.924632,284.707752,692.093891,0.0
1,submission_v2_robust,1,50000,83.892877,124.212128,174.953409,326.587276,632.933597,2258.126486,0.0
2,submission_v2_single,1,50000,83.892877,124.212128,174.953409,326.587276,632.933597,2258.126486,0.0
3,submission_v2_2_quick_robust,1,50000,80.361114,51.777235,151.844838,171.282026,213.348234,377.993684,0.0
4,submission_v2_2_quick_challenger,1,50000,49.871307,92.254298,128.711023,201.639664,436.873694,4110.736816,0.0


In [4]:
# Chargement train/test + schema submission
train, test = v2.load_train_test(ROOT / "data")
id_col, target_col, sample_sub = wf.detect_submission_schema(test, root=ROOT)
print("train/test:", train.shape, test.shape)
print("id_col:", id_col, "| target_col:", target_col, "| sample_sub:", None if sample_sub is None else sample_sub.shape)


train/test: (50000, 33) (50000, 28)
id_col: index | target_col: pred | sample_sub: None


## 3) Reproduction contrôlée du track Tweedie direct (random KFold)

On reproduit l'idée du script standalone (CatBoost Tweedie direct + alpha LS),
puis on la compare ensuite aux splits robustes V2.


In [5]:
direct_random = {
    "registry_df": pd.DataFrame(),
    "best_row": None,
    "best_payload": None,
    "payloads_by_candidate": {},
}

if RUN_DIRECT_TWEEDIE_REPRO:
    direct_random = wf.run_direct_tweedie_random_kfold(
        train,
        test,
        id_col=id_col,
        variance_powers=TWEEDIE_POWERS,
        n_splits_random=N_SPLITS_RANDOM,
        seed=SEED,
        iterations=CATBOOST_ITER,
        learning_rate=CATBOOST_LR,
        depth=CATBOOST_DEPTH,
        od_wait=OD_WAIT,
        verbose=False,
        out_dir=ARTIFACT_V23,
    )
    display(direct_random["registry_df"])
    print("best random tweedie:", direct_random["best_row"])
else:
    print("RUN_DIRECT_TWEEDIE_REPRO=False -> skip")


,track,candidate_id,run_id,cv_scheme,variance_power,alpha_ls,scale_multiplier,blend_weight,baseline_blend_source,rmse_local,...,n_valid_oof,pred_test_array,pred_q90_oof,pred_q99_oof,pred_q90_test,pred_q99_test,q90_test_over_oof,q99_test_over_oof,std_test_over_oof,distribution_collapse_flag
0,direct_tweedie_randomkfold,direct_randomkfold_p1.2,direct_tweedie|catboost|random_kfold|p1.2|seed...,random_kfold,1.2,1.022771,1.0,NaN,None,542.008281,...,50000,"[46.53263227907076, 39.62442217002603, 159.225...",178.524856,240.322954,176.419343,232.356814,0.988206,0.966852,0.983000,0
1,direct_tweedie_randomkfold,direct_randomkfold_p1.4,direct_tweedie|catboost|random_kfold|p1.4|seed...,random_kfold,1.4,1.044283,1.0,NaN,None,542.057672,...,50000,"[47.579018803145445, 41.562697580653825, 166.4...",177.668069,233.145324,175.900162,222.067265,0.990049,0.952484,0.984555,0


best random tweedie: {'track': 'direct_tweedie_randomkfold', 'candidate_id': 'direct_randomkfold_p1.2', 'run_id': 'direct_tweedie|catboost|random_kfold|p1.2|seed42|alpha_ls', 'cv_scheme': 'random_kfold', 'variance_power': 1.2, 'alpha_ls': 1.022770575543085, 'scale_multiplier': 1.0, 'blend_weight': nan, 'baseline_blend_source': None, 'rmse_local': 542.0082814847009, 'rmse_primary_time': nan, 'rmse_secondary_group': nan, 'rmse_aux_blocked5': nan, 'rmse_split_std': nan, 'q95_ratio_pos': 0.050638284588775936, 'q99_ratio_pos': 0.03735967450186192, 'rmse_prime_top1pct': 4507.033397559073, 'distribution_alignment_score': -0.4244452662192294, 'distribution_alignment_penalty': 0.4244452662192294, 'dominant_gap_hypothesis': None, 'selection_status': 'candidate', 'n_valid_oof': 50000, 'pred_test_array': array([ 46.53263228,  39.62442217, 159.22505678, ...,  95.92022828,
       125.84408601, 164.10142957], shape=(50000,)), 'pred_q90_oof': 178.52485626431456, 'pred_q99_oof': 240.32295445702673, 'pr

In [6]:
# Optionnel: comparaison avec un CSV step7 externe (si fourni)
external_step7_summary = None
if EXTERNAL_STEP7_SUBMISSION_PATH:
    p = Path(EXTERNAL_STEP7_SUBMISSION_PATH)
    if p.exists():
        ext = pd.read_csv(p)
        pred_col = "pred" if "pred" in ext.columns else (ext.columns[1] if ext.shape[1] > 1 else None)
        if pred_col is not None:
            s = pd.to_numeric(ext[pred_col], errors="coerce").fillna(0.0).clip(lower=0.0)
            external_step7_summary = pd.DataFrame([{
                "name": p.name,
                "n": int(len(s)),
                "mean": float(s.mean()),
                "std": float(s.std(ddof=0)),
                "q90": float(s.quantile(0.90)),
                "q95": float(s.quantile(0.95)),
                "q99": float(s.quantile(0.99)),
                "max": float(s.max()),
                "share_zero": float((s <= 0).mean()),
            }])
            display(external_step7_summary)
        else:
            print("CSV externe sans colonne de prediction exploitable.")
    else:
        print("EXTERNAL_STEP7_SUBMISSION_PATH non trouvé:", p)


## 4) Évaluation robuste du même Tweedie direct (splits V2)

But:
- distinguer l'effet **approche direct Tweedie** de l'effet **random KFold + scale public-LB**.


In [7]:
direct_v2_df = pd.DataFrame()
direct_v2_payloads = {"per_run_split": {}, "train_y": None}

powers_for_v2_eval = []
if len(direct_random.get("registry_df", pd.DataFrame())):
    powers_for_v2_eval = (
        direct_random["registry_df"]
        .sort_values(["rmse_local", "distribution_alignment_penalty"], na_position="last")["variance_power"]
        .astype(float)
        .drop_duplicates()
        .head(TOP_POWERS_FOR_V2_EVAL)
        .tolist()
    )
if not powers_for_v2_eval:
    powers_for_v2_eval = TWEEDIE_POWERS[:TOP_POWERS_FOR_V2_EVAL]

if RUN_DIRECT_TWEEDIE_V2_SPLITS:
    direct_v2_df, _payloads = wf.run_direct_tweedie_v2_splits(
        train,
        test,
        id_col=id_col,
        variance_powers=powers_for_v2_eval,
        seed=SEED,
        iterations=CATBOOST_ITER,
        learning_rate=CATBOOST_LR,
        depth=CATBOOST_DEPTH,
        od_wait=OD_WAIT,
        verbose=False,
        return_payloads=True,
    )
    direct_v2_payloads = dict(_payloads.get("per_run_split", {}))
    direct_v2_payloads["train_y"] = _payloads.get("train_y")
    display(direct_v2_df.sort_values(["candidate_id", "cv_scheme"]))
else:
    print("RUN_DIRECT_TWEEDIE_V2_SPLITS=False -> skip")


,track,candidate_id,run_id,cv_scheme,variance_power,alpha_ls,scale_multiplier,blend_weight,baseline_blend_source,rmse_local,...,pred_q90_oof,pred_q99_oof,pred_q90_test,pred_q99_test,q90_test_over_oof,q99_test_over_oof,std_test_over_oof,distribution_collapse_flag,rmse_gap_secondary,rmse_gap_aux
2,direct_tweedie_v2splits,direct_v2splits_p1.2,direct_tweedie|catboost|v2splits|p1.2|seed42|a...,aux_blocked5,1.2,1.024949,1.0,NaN,None,541.951529,...,178.680676,247.449725,177.565325,236.838848,0.993758,0.957119,0.981193,0.0,NaN,NaN
3,direct_tweedie_v2splits,direct_v2splits_p1.2,direct_tweedie|catboost|v2splits|p1.2|seed42|a...,multi,1.2,1.024949,1.0,NaN,None,542.434808,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-0.523004,-0.483279
0,direct_tweedie_v2splits,direct_v2splits_p1.2,direct_tweedie|catboost|v2splits|p1.2|seed42|a...,primary_time,1.2,1.068133,1.0,NaN,None,542.434808,...,177.511605,245.407479,175.801768,231.607408,0.990368,0.943767,0.976700,0.0,NaN,NaN
1,direct_tweedie_v2splits,direct_v2splits_p1.2,direct_tweedie|catboost|v2splits|p1.2|seed42|a...,secondary_group,1.2,1.022127,1.0,NaN,None,541.911804,...,179.270423,245.877291,177.570714,235.763000,0.990519,0.958864,0.980409,0.0,NaN,NaN


In [8]:
# Registry direct Tweedie combiné (random KFold + splits V2)
direct_registry_combined = pd.concat(
    [
        direct_random.get("registry_df", pd.DataFrame()),
        direct_v2_df,
    ],
    ignore_index=True,
    sort=False,
)
if len(direct_registry_combined):
    direct_registry_combined.drop(columns=["pred_test_array"], errors="ignore").to_csv(ARTIFACT_V23 / "direct_tweedie_cv_registry.csv", index=False)
    print("saved:", ARTIFACT_V23 / "direct_tweedie_cv_registry.csv")
    display(direct_registry_combined.head(20))
else:
    print("No direct Tweedie registry rows.")


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_3_dualtrack_quick\direct_tweedie_cv_registry.csv


,track,candidate_id,run_id,cv_scheme,variance_power,alpha_ls,scale_multiplier,blend_weight,baseline_blend_source,rmse_local,...,pred_q90_oof,pred_q99_oof,pred_q90_test,pred_q99_test,q90_test_over_oof,q99_test_over_oof,std_test_over_oof,distribution_collapse_flag,rmse_gap_secondary,rmse_gap_aux
0,direct_tweedie_randomkfold,direct_randomkfold_p1.2,direct_tweedie|catboost|random_kfold|p1.2|seed...,random_kfold,1.2,1.022771,1.0,NaN,None,542.008281,...,178.524856,240.322954,176.419343,232.356814,0.988206,0.966852,0.983000,0.0,NaN,NaN
1,direct_tweedie_randomkfold,direct_randomkfold_p1.4,direct_tweedie|catboost|random_kfold|p1.4|seed...,random_kfold,1.4,1.044283,1.0,NaN,None,542.057672,...,177.668069,233.145324,175.900162,222.067265,0.990049,0.952484,0.984555,0.0,NaN,NaN
2,direct_tweedie_v2splits,direct_v2splits_p1.2,direct_tweedie|catboost|v2splits|p1.2|seed42|a...,primary_time,1.2,1.068133,1.0,NaN,None,542.434808,...,177.511605,245.407479,175.801768,231.607408,0.990368,0.943767,0.976700,0.0,NaN,NaN
3,direct_tweedie_v2splits,direct_v2splits_p1.2,direct_tweedie|catboost|v2splits|p1.2|seed42|a...,secondary_group,1.2,1.022127,1.0,NaN,None,541.911804,...,179.270423,245.877291,177.570714,235.763000,0.990519,0.958864,0.980409,0.0,NaN,NaN
4,direct_tweedie_v2splits,direct_v2splits_p1.2,direct_tweedie|catboost|v2splits|p1.2|seed42|a...,aux_blocked5,1.2,1.024949,1.0,NaN,None,541.951529,...,178.680676,247.449725,177.565325,236.838848,0.993758,0.957119,0.981193,0.0,NaN,NaN
5,direct_tweedie_v2splits,direct_v2splits_p1.2,direct_tweedie|catboost|v2splits|p1.2|seed42|a...,multi,1.2,1.024949,1.0,NaN,None,542.434808,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-0.523004,-0.483279


## 5) Scale sweep + blend sweep (contrôlés)

On applique les heuristiques public-LB (scale / blend), mais on les journalise explicitement avec des métriques locales et des flags de risque.


In [9]:
scale_sweep_df = wf.compute_scale_sweep(
    random_result=direct_random,
    scale_sweep=SCALE_SWEEP,
    v2_split_payloads=direct_v2_payloads,
) if len(direct_random.get("registry_df", pd.DataFrame())) else pd.DataFrame()

if len(scale_sweep_df):
    scale_sweep_df.drop(columns=["pred_test_array"], errors="ignore").to_csv(ARTIFACT_V23 / "direct_tweedie_scale_sweep.csv", index=False)
    print("saved:", ARTIFACT_V23 / "direct_tweedie_scale_sweep.csv")
    display(scale_sweep_df.sort_values(['rmse_local', 'distribution_alignment_penalty'], na_position='last'))
else:
    print("Scale sweep unavailable.")


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_3_dualtrack_quick\direct_tweedie_scale_sweep.csv


,track,candidate_id,parent_candidate_id,cv_scheme,variance_power,alpha_ls,scale_multiplier,blend_weight,baseline_blend_source,rmse_local,...,pred_q90_oof,pred_q99_oof,pred_q90_test,pred_q99_test,q90_test_over_oof,q99_test_over_oof,std_test_over_oof,distribution_alignment_penalty,distribution_alignment_score,distribution_collapse_flag
0,direct_tweedie_scale,direct_scale_p1.2_x1.0,direct_randomkfold_p1.2,random_kfold,1.2,1.022771,1.0,NaN,None,542.008281,...,178.524856,240.322954,176.419343,232.356814,0.988206,0.966852,0.983,0.424445,-0.424445,0
1,direct_tweedie_scale,direct_scale_p1.2_x1.1,direct_randomkfold_p1.2,random_kfold,1.2,1.022771,1.1,NaN,None,542.140833,...,196.377342,264.355250,194.061277,255.592496,0.988206,0.966852,0.983,0.424445,-0.424445,0
2,direct_tweedie_scale,direct_scale_p1.2_x1.2,direct_randomkfold_p1.2,random_kfold,1.2,1.022771,1.2,NaN,None,542.538295,...,214.229828,288.387545,211.703211,278.828177,0.988206,0.966852,0.983,0.424445,-0.424445,0
3,direct_tweedie_scale,direct_scale_p1.2_x1.3,direct_randomkfold_p1.2,random_kfold,1.2,1.022771,1.3,NaN,None,543.200084,...,232.082313,312.419841,229.345145,302.063859,0.988206,0.966852,0.983,0.424445,-0.424445,0


In [10]:
blend_sweep_df = pd.DataFrame()
if RUN_BLEND_SWEEP and len(scale_sweep_df):
    blend_sweep_df = wf.compute_blend_sweep(
        ctx=ctx,
        scale_df=scale_sweep_df,
        random_result=direct_random,
        v2_splits_payloads=direct_v2_payloads,
        blend_weights=BLEND_WEIGHTS,
    )
    if len(blend_sweep_df):
        blend_sweep_df.drop(columns=["pred_test_array"], errors="ignore").to_csv(ARTIFACT_V23 / "direct_tweedie_blend_sweep.csv", index=False)
        print("saved:", ARTIFACT_V23 / "direct_tweedie_blend_sweep.csv")
        display(blend_sweep_df.sort_values(['rmse_local', 'blend_weight'], na_position='last'))
    else:
        print("Blend sweep unavailable (baseline submission/OOF missing or mismatch).")
else:
    print("Blend sweep skipped.")


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_3_dualtrack_quick\direct_tweedie_blend_sweep.csv


,track,candidate_id,cv_scheme,variance_power,alpha_ls,scale_multiplier,blend_weight,baseline_blend_source,rmse_local,rmse_primary_time,...,q95_ratio_pos,q99_ratio_pos,rmse_prime_top1pct,comparability_partial,pred_test_array,selection_status,pred_q90_test,pred_q99_test,pred_std_test,distribution_collapse_flag
0,blend,blend_direct_scale_w0.2,test_only,1.2,1.022771,1.0,0.2,submission_v1.csv,NaN,NaN,...,NaN,NaN,NaN,1,"[72.73543826606078, 55.915374836606674, 191.15...",candidate,172.280363,269.919449,60.662041,0.0
1,blend,blend_direct_scale_w0.4,test_only,1.2,1.022771,1.0,0.4,submission_v1.csv,NaN,NaN,...,NaN,NaN,NaN,1,"[66.18473676931328, 51.842636669961514, 183.17...",candidate,172.552598,257.525071,59.445619,0.0
2,blend,blend_direct_scale_w0.6,test_only,1.2,1.022771,1.0,0.6,submission_v1.csv,NaN,NaN,...,NaN,NaN,NaN,1,"[59.63403527256577, 47.769898503316355, 175.19...",candidate,173.233776,245.285848,58.727839,0.0
3,blend,blend_direct_scale_w0.8,test_only,1.2,1.022771,1.0,0.8,submission_v1.csv,NaN,NaN,...,NaN,NaN,NaN,1,"[53.08333377581826, 43.69716033667119, 167.208...",candidate,174.861182,236.790591,58.527050,0.0


## 6) Diagnostic overfit / OOD / queue (dual-track)

On fusionne:
- le diagnostic V2.2 quick,
- le track Tweedie direct (random + V2 splits),
- les sweeps `scale` / `blend`.


In [11]:
# Candidats V2.3 pour sélection (on garde les agrégats multi-splits pour le track direct robuste)
direct_v2_multi = direct_v2_df[direct_v2_df.get('cv_scheme', pd.Series(dtype=object)).astype(str).eq('multi')].copy() if len(direct_v2_df) else pd.DataFrame()
direct_random_registry = direct_random.get("registry_df", pd.DataFrame()).copy()

v23_candidates_df = pd.concat(
    [direct_v2_multi, direct_random_registry, scale_sweep_df, blend_sweep_df],
    ignore_index=True,
    sort=False,
)

diagnosis_df, diagnosis_summary = wf.classify_gap_cause_dualtrack(
    ctx=ctx,
    v23_candidates_df=v23_candidates_df,
    kaggle_public_rmse_user=KAGGLE_PUBLIC_RMSE_USER,
)

diagnosis_df.to_csv(ARTIFACT_V23 / "overfit_ood_queue_diagnosis.csv", index=False)
(ARTIFACT_V23 / "overfit_ood_queue_summary.json").write_text(json.dumps(diagnosis_summary, indent=2), encoding="utf-8")

print("saved:", ARTIFACT_V23 / "overfit_ood_queue_diagnosis.csv")
print("saved:", ARTIFACT_V23 / "overfit_ood_queue_summary.json")
display(diagnosis_df.head(20))
print("diagnosis_summary:", diagnosis_summary)


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_3_dualtrack_quick\overfit_ood_queue_diagnosis.csv
saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_3_dualtrack_quick\overfit_ood_queue_summary.json


,candidate_id,track,cv_instability_flag,ood_risk_flag,tail_undercoverage_flag,tail_overcorrection_flag,public_lb_heuristic_flag,kaggle_gap_hypothesis,evidence_summary
0,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,v2_two_part,0,0,1,0,0,Gap domine par sous-modelisation de la queue,V2 shortlist; q99=0.329; split_std=0.729
1,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,v2_two_part,0,0,1,0,0,Gap domine par sous-modelisation de la queue,V2 shortlist; q99=0.318; split_std=0.797
2,base_v2|catboost|two_part_classic|cb_v2_c1|42|...,v2_two_part,0,0,1,0,0,Gap domine par sous-modelisation de la queue,V2 shortlist; q99=0.310; split_std=0.844
3,base_v2|xgboost|two_part_tweedie|xgb_v2_c1|42|...,v2_two_part,0,0,1,0,0,Gap domine par sous-modelisation de la queue,V2 shortlist; q99=0.325; split_std=0.853
4,base_v2|lightgbm|two_part_classic|lgb_v2_c1|42...,v2_two_part,0,0,1,0,0,Gap domine par sous-modelisation de la queue,V2 shortlist; q99=0.290; split_std=0.692
5,direct_v2splits_p1.2,direct_tweedie_v2splits,0,0,1,0,0,Gap domine par sous-modelisation de la queue,rmse_primary_time=542.435; rmse_secondary_grou...
6,direct_randomkfold_p1.2,direct_tweedie_randomkfold,0,0,1,0,1,Gap domine par sous-modelisation de la queue,q99_ratio_pos=0.037
7,direct_randomkfold_p1.4,direct_tweedie_randomkfold,0,0,1,0,1,Gap domine par sous-modelisation de la queue,q99_ratio_pos=0.036
8,direct_scale_p1.2_x1.0,direct_tweedie_scale,0,0,1,0,1,Gap domine par sous-modelisation de la queue,rmse_primary_time=542.435; q99_ratio_pos=0.037
9,direct_scale_p1.2_x1.1,direct_tweedie_scale,0,0,1,0,1,Gap domine par sous-modelisation de la queue,rmse_primary_time=542.566; q99_ratio_pos=0.041


diagnosis_summary: {'kaggle_public_rmse_user': 1365.5, 'note_metric_non_comparable': True, 'n_candidates_analyzed': 16, 'dominant_hypothesis_counts': {'Gap domine par sous-modelisation de la queue': 12, 'Overfitting non prouve': 4}, 'dominant_hypothesis_top_run': 'Gap domine par sous-modelisation de la queue', 'initial_working_hypothesis': 'Mixte (OOD + queue)', 'overfitting_cv_proven': False}


## 7) Sélection finale V2.3 (robust + LB challenger) et génération des soumissions

Règle:
- `robust` priorise les garde-fous multi-splits,
- `lb_challenger` accepte plus de risque si la logique est explicitement LB-driven.


In [12]:
baseline_sub_df, baseline_sub_name = wf._extract_submission_baseline(ctx)
selected_df = wf.select_dualtrack_submissions(
    v23_candidates_df,
    baseline_submission_df=baseline_sub_df if len(baseline_sub_df) else None,
    baseline_name=baseline_sub_name,
    id_col=id_col,
)

display(selected_df[[c for c in ['role','candidate_id','track','risk_tag','rmse_local','rmse_primary_time','rmse_secondary_group','rmse_aux_blocked5','q99_ratio_pos','selection_score_dualtrack'] if c in selected_df.columns]])

outputs = wf.materialize_dualtrack_outputs(
    ctx=ctx,
    test_df=test,
    id_col=id_col,
    target_col=target_col,
    sample_submission_df=sample_sub,
    selected_df=selected_df,
    direct_random_result=direct_random,
    direct_v2_payloads=direct_v2_payloads,
    run_shakeup_quick=RUN_SHAKEUP_QUICK,
    n_sim_shakeup=N_SIM_SHAKEUP,
    seed=SEED,
    out_dir=ARTIFACT_V23,
)
print("submission paths:", outputs.get("submission_paths", {}))


,role,candidate_id,track,risk_tag,rmse_local,rmse_primary_time,rmse_secondary_group,rmse_aux_blocked5,q99_ratio_pos,selection_score_dualtrack
0,robust,direct_v2splits_p1.2,direct_tweedie_v2splits,robust,542.434808,542.434808,541.911804,541.951529,0.038417,562.323562
1,lb_challenger,blend_direct_scale_w0.2,blend,public_private_risk,NaN,NaN,NaN,NaN,NaN,NaN


submission paths: {'robust': WindowsPath('c:/Users/icemo/Downloads/Calcul-prime-d-assurance/artifacts/v2_3_dualtrack_quick/submission_v2_3_robust.csv'), 'lb_challenger': WindowsPath('c:/Users/icemo/Downloads/Calcul-prime-d-assurance/artifacts/v2_3_dualtrack_quick/submission_v2_3_lb_challenger.csv')}


In [13]:
# 7bis) OOF bridge compare (V1 / V2 selected / direct Tweedie)
oof_compare_bridge = wf.build_oof_compare_bridge(
    ctx=ctx,
    direct_random_result=direct_random,
    direct_v2_payloads=direct_v2_payloads,
    out_dir=ARTIFACT_V23,
)
if len(oof_compare_bridge):
    print("saved:", ARTIFACT_V23 / "oof_compare_bridge.parquet", oof_compare_bridge.shape)
    display(oof_compare_bridge.head(10))
else:
    print("OOF bridge compare not generated (insufficient sources).")


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_3_dualtrack_quick\oof_compare_bridge.parquet (50000, 6)


,row_idx,y_true,pred_v1_best,pred_v2_selected,pred_direct_random_best,pred_direct_primary
0,0,0.00,NaN,0.0,102.336681,NaN
1,1,0.00,NaN,0.0,189.155955,NaN
2,2,0.00,NaN,0.0,137.839004,NaN
3,3,0.00,NaN,0.0,38.248683,NaN
4,4,0.00,NaN,0.0,110.828644,NaN
5,5,0.00,NaN,0.0,28.061013,NaN
6,6,0.00,NaN,0.0,154.609350,NaN
7,7,0.00,NaN,0.0,97.799096,NaN
8,8,927.16,NaN,0.0,70.410093,NaN
9,9,0.00,NaN,0.0,126.288339,NaN


In [14]:
# 7ter) pred_distribution_compare.csv consolidé (existants + nouveaux V2.3)
pred_distribution_new_df = outputs.get("pred_distribution_compare", pd.DataFrame())
pred_distribution_compare_df = pd.concat(
    [pred_distribution_existing_df, pred_distribution_new_df],
    ignore_index=True,
    sort=False,
)
if external_step7_summary is not None:
    pred_distribution_compare_df = pd.concat([pred_distribution_compare_df, external_step7_summary], ignore_index=True, sort=False)

if len(pred_distribution_compare_df):
    pred_distribution_compare_df.to_csv(ARTIFACT_V23 / "pred_distribution_compare.csv", index=False)
    print("saved:", ARTIFACT_V23 / "pred_distribution_compare.csv")
    display(pred_distribution_compare_df)


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_3_dualtrack_quick\pred_distribution_compare.csv


,name,exists,n,mean,std,q90,q95,q99,max,share_zero,...,pred_share_nonzero,pred_identical_share,pred_identical_share_nonzero,pred_q99_q90_ratio,pred_n_nonzero,pred_q90_nonzero,pred_q99_nonzero,pred_q99_q90_ratio_nonzero,collapse_use_nonzero_support,distribution_collapse_flag
0,submission_v1,1.0,50000,97.606838,62.347925,172.878272,201.924632,284.707752,692.093891,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,submission_v2_robust,1.0,50000,83.892877,124.212128,174.953409,326.587276,632.933597,2258.126486,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,submission_v2_single,1.0,50000,83.892877,124.212128,174.953409,326.587276,632.933597,2258.126486,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,submission_v2_2_quick_robust,1.0,50000,80.361114,51.777235,151.844838,171.282026,213.348234,377.993684,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,submission_v2_2_quick_challenger,1.0,50000,49.871307,92.254298,128.711023,201.639664,436.873694,4110.736816,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,50000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,0.00004,0.00004,1.317435,50000.0,175.801768,231.607408,1.317435,0.0,0.0
6,NaN,NaN,50000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,0.00004,0.00004,1.566745,50000.0,172.280363,269.919449,1.566745,0.0,0.0


In [15]:
# 8) Rapport de décision final V2.3
decision_path = wf.write_submission_decision_report(
    ctx=ctx,
    kaggle_public_rmse_user=KAGGLE_PUBLIC_RMSE_USER,
    bridge_summary_df=bridge_summary_df,
    diagnosis_summary=diagnosis_summary,
    diagnosis_df=diagnosis_df,
    selected_df=selected_df,
    out_dir=ARTIFACT_V23,
)
print("saved:", decision_path)
print(decision_path.read_text(encoding='utf-8')[:4000])


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_3_dualtrack_quick\submission_decision_v2_3_dualtrack.md
# Submission decision V2.3 Dual-Track Quick

## 1) Contexte
- Kaggle public (utilisateur): ~1365.500
- OOF local et Kaggle public ne sont pas directement comparables.
- Ce cycle compare un track robuste V2 et un track direct Tweedie CatBoost (random KFold + scale/blend).

## 2) Diagnostic du gap (resume)
- Hypothese dominante (dual-track): Gap domine par sous-modelisation de la queue
- Comptage hypotheses: {'Gap domine par sous-modelisation de la queue': 12, 'Overfitting non prouve': 4}
- Overfitting CV: non conclu sans preuve inter-splits.
- Public LB peut reagir positivement a un scaling global sans garantie privee.

## 3) Bridge V1 / V2 / V2.2 (local)

| model_version         | track       | run_id                                                                                       |   rmse_primary_time |   rmse_secondary_group |   rmse_aux_blocked5 |   q95_

## Checklist des artefacts V2.3 attendus

- `run_bridge_summary.csv`
- `direct_tweedie_cv_registry.csv`
- `direct_tweedie_scale_sweep.csv`
- `direct_tweedie_blend_sweep.csv` (si blend possible)
- `overfit_ood_queue_diagnosis.csv`
- `overfit_ood_queue_summary.json`
- `oof_compare_bridge.parquet`
- `pred_distribution_compare.csv`
- `shakeup_bridge_top.parquet` (si `RUN_SHAKEUP_QUICK=True` et OOF disponible)
- `submission_v2_3_robust.csv`
- `submission_v2_3_lb_challenger.csv`
- `submission_decision_v2_3_dualtrack.md`
